# 11 · Multi-Agent Orchestration (a supervisor and its specialists)

**Where we are in the stack:** the **orchestration plane**. The punchline of this notebook is
one sentence: **an agent can be a tool**. A "multi-agent system" is the same loop from
notebook 02, where one of the entries in the capability table happens to run *another* loop.

The NOC analogy: a **dispatcher** (tier-1) takes the ticket, decides which **tier-2 team** -
systems, platform, or data - should own it, hands it off, and assembles the final reply. The
dispatcher does not read logs; the log team does not design the reply.

```
                       +-> codebase specialist (04's tools: ls/cat)
user -> SUPERVISOR ----+-> log specialist      (05's tools: tail/grep/count)
        (a loop whose  +-> database specialist (06's tools: SELECT-only SQL)
         tools are agents)        each specialist = the same loop, its own context
```

Why bother, beyond routing? **Context isolation.** Each specialist works in its own fresh
transcript with only its own tools - the supervisor's context stays small, and a specialist
cannot be confused by another domain's noise. (Same reason you segment a network.)

> Needs a tool-capable model (see notebook 02). Sample logs and the inventory DB are
> generated below - self-contained as always.

In [ ]:
# --- Provider config: works with OpenAI, OpenRouter, or a local OpenAI-compatible server ---
import os
from openai import OpenAI

# Load settings from a .env file if present (falls back to existing env vars).
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    import os
    if os.path.exists(".env"):
        for _line in open(".env"):
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip())


# Pick ONE setup by exporting these env vars before launching Jupyter.
#
#   OpenAI:     OPENAI_BASE_URL=https://api.openai.com/v1   MODEL=gpt-4o-mini
#   OpenRouter: OPENAI_BASE_URL=https://openrouter.ai/api/v1 MODEL=openai/gpt-4o-mini
#   Local:      OPENAI_BASE_URL=http://localhost:11434/v1    MODEL=qwen2.5:7b   (Ollama)
#               (use 'qwen2.5' / 'llama3.1' etc. - a 1B model is great for chat but
#                usually too weak to drive tool-calling reliably.)

BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
API_KEY  = os.environ.get("OPENAI_API_KEY", "set-me")   # any non-empty string for local servers
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

# Behind a TLS-intercepting firewall/proxy, HTTPS cert verification can fail.
# Set VERIFY_SSL=false in .env to skip it: we hand the OpenAI SDK a custom
# httpx client with verification turned off. Leave it true everywhere else.
import httpx
VERIFY_SSL = os.environ.get("VERIFY_SSL", "true").strip().lower() not in ("false", "0", "no")
http_client = httpx.Client(verify=VERIFY_SSL)
if not VERIFY_SSL:
    import warnings
    warnings.filterwarnings("ignore")
    print("\u26a0\ufe0f  SSL verification DISABLED (VERIFY_SSL=false) \u2014 use only on a trusted network")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY, http_client=http_client)
print("endpoint:", BASE_URL, "| model:", MODEL)

## 1. The loop, one last time - now as a factory

We have copied notebook 02's FSM four times across this repo. The repetition earned us the
right to do this: parameterize it. `run_loop` takes a system prompt + capability table and
returns an answer. **Every agent in this notebook - specialists and supervisor alike - is
one call to this function.**

In [ ]:
import json

def run_loop(system, tools, registry, question, label="agent", max_iterations=10, temperature=0):
    """Notebook 02's FSM, parameterized: one loop to run every agent in this notebook."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_iterations + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, temperature=temperature,
        )
        msg = resp.choices[0].message

        # TERMINATION: no tool requested -> final answer.
        if not msg.tool_calls:
            return msg.content

        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            print(f"  [{label} | step {step}] {name}({args})")
            try:
                result = registry[name](**args)              # <-- WE run it
            except Exception as e:
                result = {"error": str(e)}
            messages.append({
                "role": "tool", "tool_call_id": tc.id,
                "content": json.dumps(result)[:4000],
            })

    return f"[{label}] Stopped: hit max_iterations (TTL expired)."

## 2. Generate the specialists' worlds

Compact versions of the sample data from notebooks 05 and 06 (skipped if already present).

In [ ]:
import os, random, sqlite3

# --- sample logs (notebook 05) ---
LOG_ROOT = os.path.abspath("sample_logs")
if not os.path.isdir(LOG_ROOT):
    os.makedirs(LOG_ROOT)
    random.seed(7)
    levels = ["INFO"]*7 + ["WARN"]*2 + ["ERROR"]
    msgs = {
        "INFO":  ["request handled", "cache hit", "user login", "job complete"],
        "WARN":  ["high latency", "retry scheduled", "cache miss storm"],
        "ERROR": ["connection refused", "timeout to db", "null pointer", "disk full"],
    }
    for fname, n in [("api.log", 60), ("worker.log", 40)]:
        with open(os.path.join(LOG_ROOT, fname), "w") as f:
            for i in range(n):
                lvl = random.choice(levels)
                f.write(f"2026-06-23T04:{i%60:02d}:00 {lvl} {fname[:-4]}: {random.choice(msgs[lvl])}\n")

# --- inventory database (notebook 06) ---
DB_PATH = os.path.abspath("inventory.db")
if not os.path.exists(DB_PATH):
    con = sqlite3.connect(DB_PATH)
    con.executescript("""
    CREATE TABLE devices (
        id INTEGER PRIMARY KEY, hostname TEXT, role TEXT, site TEXT, uptime_days INTEGER);
    CREATE TABLE interfaces (
        id INTEGER PRIMARY KEY, device_id INTEGER, name TEXT,
        oper_status TEXT, speed_gbps INTEGER, errors INTEGER);
    INSERT INTO devices (hostname, role, site, uptime_days) VALUES
        ('leaf-01','leaf','blr', 120), ('leaf-02','leaf','blr', 12),
        ('spine-01','spine','blr', 300), ('spine-02','spine','del', 5);
    INSERT INTO interfaces (device_id, name, oper_status, speed_gbps, errors) VALUES
        (1,'et-0/0/1','up',10,0), (1,'et-0/0/2','down',10,42),
        (2,'et-0/0/1','up',25,3), (3,'et-0/0/1','up',100,0),
        (3,'et-0/0/2','up',100,7), (4,'et-0/0/1','down',25,15);
    """)
    con.commit(); con.close()

FS_ROOT = os.path.abspath(".")   # the codebase specialist explains THIS repo
print("logs:", LOG_ROOT, "| db:", DB_PATH, "| code:", FS_ROOT)

## 3. Three specialists = three capability tables

Compact versions of notebooks 04, 05 and 06's tools, with the **same guards**: filesystem
and logs sandboxed under their roots, SQL gated to a single `SELECT`. Each specialist gets
its own `(system, tools, registry)` triple - nothing is shared between them except
`run_loop`.

In [ ]:
import subprocess

def _safe(root, path):
    full = os.path.abspath(os.path.join(root, path))
    if full != root and not full.startswith(root + os.sep):
        raise ValueError(f"path '{path}' escapes the sandbox")
    return full

# --- codebase tools (notebook 04, condensed) ---
def list_dir(path="."):
    """ls: list files/dirs at `path` (relative to the repo root)."""
    full = _safe(FS_ROOT, path)
    if not os.path.isdir(full):
        return {"error": f"not a directory: {path}"}
    return {"path": path, "entries": sorted(
        n + ("/" if os.path.isdir(os.path.join(full, n)) else "")
        for n in os.listdir(full) if n not in (".git", ".venv"))}

def read_file(path, max_chars=6000):
    """cat: return the text of one file (truncated)."""
    full = _safe(FS_ROOT, path)
    if not os.path.isfile(full):
        return {"error": f"not a file: {path}"}
    text = open(full, errors="replace").read()
    return {"path": path, "truncated": len(text) > max_chars, "text": text[:max_chars]}

FS_SYSTEM = ("You are a codebase specialist. Answer ONLY by inspecting files with your "
             "tools. Cite file paths. Be concise.")
FS_TOOLS = [
    {"type": "function", "function": {"name": "list_dir",
        "description": "List files and directories at a path (like ls).",
        "parameters": {"type": "object",
            "properties": {"path": {"type": "string", "description": "dir, default '.'"}},
            "required": []}}},
    {"type": "function", "function": {"name": "read_file",
        "description": "Read one file's text (like cat).",
        "parameters": {"type": "object",
            "properties": {"path": {"type": "string"}},
            "required": ["path"]}}},
]
FS_REG = {"list_dir": list_dir, "read_file": read_file}

In [ ]:
# --- log tools (notebook 05, condensed) ---
def tail(path, n=15):
    """tail -n: last n lines of a log file."""
    full = _safe(LOG_ROOT, path)
    if not os.path.isfile(full):
        return {"error": f"not a file: {path}"}
    return {"path": path, "lines": open(full, errors="replace").read().splitlines()[-n:]}

def grep_logs(pattern, max_matches=40):
    """grep -rn: search the logs for a pattern."""
    out = subprocess.run(["grep", "-rIn", "--", pattern, LOG_ROOT],
                         capture_output=True, text=True).stdout
    matches = [l.replace(LOG_ROOT + os.sep, "") for l in out.splitlines()[:max_matches]]
    return {"pattern": pattern, "matches": matches, "count": len(matches)}

def list_logs():
    """ls: list the available log files."""
    return {"entries": sorted(os.listdir(LOG_ROOT))}

LOG_SYSTEM = ("You are a log-triage specialist. Investigate ONLY via your log tools. "
              "Quote example lines. Be concise.")
LOG_TOOLS = [
    {"type": "function", "function": {"name": "list_logs",
        "description": "List the available log files.",
        "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "tail",
        "description": "Last n lines of a log file.",
        "parameters": {"type": "object",
            "properties": {"path": {"type": "string"}, "n": {"type": "integer"}},
            "required": ["path"]}}},
    {"type": "function", "function": {"name": "grep_logs",
        "description": "Search all logs for a pattern; returns file:line:text matches.",
        "parameters": {"type": "object",
            "properties": {"pattern": {"type": "string"}},
            "required": ["pattern"]}}},
]
LOG_REG = {"list_logs": list_logs, "tail": tail, "grep_logs": grep_logs}

In [ ]:
# --- database tools (notebook 06, condensed; SELECT-only gate intact) ---
def list_tables():
    """List user tables in the database."""
    con = sqlite3.connect(DB_PATH)
    rows = con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
    con.close()
    return {"tables": [r[0] for r in rows]}

def describe_table(table):
    """Columns and types of one table."""
    con = sqlite3.connect(DB_PATH)
    try:
        rows = con.execute(f"PRAGMA table_info({table})").fetchall()
    finally:
        con.close()
    return {"table": table, "columns": [{"name": r[1], "type": r[2]} for r in rows]}

def run_select(query, max_rows=50):
    """Run a single read-only SELECT and return rows. Writes are rejected."""
    q = query.strip().rstrip(";")
    low = q.lower()
    if not low.startswith("select"):
        return {"error": "only SELECT statements are allowed"}
    forbidden = (" insert ", " update ", " delete ", " drop ", " alter ",
                 " create ", " replace ", " attach ", " pragma ")
    if any(tok in f" {low} " for tok in forbidden) or ";" in q:
        return {"error": "statement contains a non-read-only or chained command"}
    con = sqlite3.connect(DB_PATH)
    try:
        cur = con.execute(q)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchmany(max_rows)
    except sqlite3.Error as e:
        con.close(); return {"error": str(e)}
    con.close()
    return {"columns": cols, "rows": [list(r) for r in rows], "row_count": len(rows)}

SQL_SYSTEM = ("You are a database specialist for the network inventory. Explore the schema "
              "before querying. Use only SELECT. Be concise.")
SQL_TOOLS = [
    {"type": "function", "function": {"name": "list_tables",
        "description": "List the tables in the database.",
        "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "describe_table",
        "description": "Columns and types of a table.",
        "parameters": {"type": "object",
            "properties": {"table": {"type": "string"}}, "required": ["table"]}}},
    {"type": "function", "function": {"name": "run_select",
        "description": "Run a single read-only SELECT query.",
        "parameters": {"type": "object",
            "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
]
SQL_REG = {"list_tables": list_tables, "describe_table": describe_table, "run_select": run_select}

# Sanity-check one tool per specialist, without any model
print(list_dir(".")["entries"][:5])
print(list_logs())
print(list_tables())

## 4. The supervisor: agents as tools

Here is the whole idea in code. Each `ask_*` function **is a complete agent run** - a fresh
context, its own tools, its own investigation - but to the supervisor it is indistinguishable
from `calculate_subnet` in notebook 02: a name, a schema, a JSON result. Delegation is just
another tool call.

In [ ]:
def ask_codebase(question):
    """Delegate to the codebase specialist (fresh context, filesystem tools)."""
    return {"specialist": "codebase", "answer": run_loop(
        FS_SYSTEM, FS_TOOLS, FS_REG, question, label="codebase")}

def ask_logs(question):
    """Delegate to the log-triage specialist (fresh context, log tools)."""
    return {"specialist": "logs", "answer": run_loop(
        LOG_SYSTEM, LOG_TOOLS, LOG_REG, question, label="logs")}

def ask_database(question):
    """Delegate to the database specialist (fresh context, SELECT-only SQL)."""
    return {"specialist": "database", "answer": run_loop(
        SQL_SYSTEM, SQL_TOOLS, SQL_REG, question, label="database")}

SUP_SYSTEM = """You are the NOC duty supervisor. You never inspect anything yourself -
you delegate to specialists and assemble their findings.
- ask_codebase: questions about this project's source code and notebooks
- ask_logs: questions about service logs, errors, failures
- ask_database: questions about the device/interface inventory
Send each specialist a precise, self-contained question. Use more than one when needed.
Synthesize a single concise answer and say which specialist(s) you used."""

SUP_TOOLS = [
    {"type": "function", "function": {"name": name,
        "description": fn.__doc__,
        "parameters": {"type": "object",
            "properties": {"question": {"type": "string",
                "description": "a precise, self-contained question for this specialist"}},
            "required": ["question"]}}}
    for name, fn in [("ask_codebase", ask_codebase), ("ask_logs", ask_logs),
                     ("ask_database", ask_database)]
]
SUP_REG = {"ask_codebase": ask_codebase, "ask_logs": ask_logs, "ask_database": ask_database}

def supervise(question):
    print("USER:", question); print("=" * 72)
    answer = run_loop(SUP_SYSTEM, SUP_TOOLS, SUP_REG, question, label="supervisor")
    print("-" * 72); print(answer)
    return answer

## 5. Run it - watch the routing

The indented trace lines show which loop is doing the work: `[supervisor]` delegating,
`[logs]` / `[database]` / `[codebase]` actually investigating.

In [ ]:
_ = supervise("Are there any database-related failures in the service logs? How many?")

In [ ]:
_ = supervise("Which devices in the inventory have a down interface, and how many errors are on those interfaces?")

### A cross-domain question (two specialists, one answer)

In [ ]:
_ = supervise(
    "Compare what the logs say about db timeouts with what the inventory database shows "
    "about down interfaces - could they be related? Summarize for a handover note."
)

## Recap

- **An agent is a tool.** The supervisor's capability table holds three functions that
  happen to run other loops. No new machinery - notebook 02's FSM, composed with itself.
- **Context isolation** is the real win: each specialist reasons over a clean transcript
  with only its own domain's tools, and the supervisor sees only their *conclusions* -
  segmentation, not one flat broadcast domain.
- The trade-offs are real too: every delegation is extra model calls (latency + cost), and
  the supervisor's answer can only be as good as the sub-answers it receives - which is why
  the next notebook is about *checking* answers.
- Frameworks (LangGraph, notebook 03) formalize this pattern as graphs of nodes with
  handoffs, retries, and shared state - same idea, production plumbing.

**Next:** how do you know any of these answers are *good*? Evals and guardrails. ->
`12_eval_and_guardrails.ipynb`